In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from sqlalchemy import create_engine


In [ ]:
df = db.run_query("SELECT * FROM geo.voterfile_042025 WHERE lat IS NOT NULL AND lon IS NOT NULL")


create_table_sql = """
create table if not exists sheets.table_name
 (
    voter_id int,
    status varchar(36),
    county varchar(3),
    first_name varchar(15),
    middle_name varchar(15),
    last_name varchar(4),
    suffix varchar(4),
    address_1 varchar(20),
    address_2 varchar(10),
    city varchar(12),
    state varchar(2),
    zip_code varchar(10),
    age int,
    dob date,
    registration_date date,
    party varchar(3),
    race varchar(1),
    gender varchar(1),
    precinct int

);
"""

In [12]:
# Centered on Miami-Dade (approx. downtown Miami)
map_center = [25.7617, -80.1918]  # Miami coordinates

In [22]:
legend_html = '''
 <div style="
     position: fixed; 
     bottom: 50px; left: 50px; width: 180px; height: 120px; 
     background-color: white;
     border:2px solid grey; z-index:9999; font-size:14px;
     padding: 10px;
     ">
     <b>Heatmap Density</b><br>
     <i style="background: #00f; width: 18px; height: 10px; display: inline-block;"></i> Low<br>
     <i style="background: #0f0; width: 18px; height: 10px; display: inline-block;"></i> Medium<br>
     <i style="background: #ff0; width: 18px; height: 10px; display: inline-block;"></i> High<br>
     <i style="background: #f00; width: 18px; height: 10px; display: inline-block;"></i> Very High
 </div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

In [36]:
# Create heatmap

m = folium.Map(location=[25.77, -80.25], zoom_start=13)
HeatMap(data=df[['lat', 'lon']].dropna(), radius=10).add_to(m)

# Add custom legend
m.get_root().html.add_child(folium.Element(legend_html))

m.save("miami_heatmap.html")

In [61]:
import geopandas as gpd
import folium
from folium import GeoJson
from folium.plugins import HeatMap
from db import get_engine  # your reusable engine
from shapely import wkt

# 1. Load geocoded points
import pandas as pd
df = pd.read_sql("select * from sheets.repo_petition rp left join geo.voterfile_042025 v on rp.voter_id = v.voter_id WHERE v.lat IS NOT NULL AND v.lon IS NOT NULL", get_engine())




# 2. Load the shapefile as GeoDataFrame from PostGIS
gdf = gpd.read_postgis(
    "SELECT * FROM geo.dade_precinct", 
    con=get_engine(), 
    geom_col="geometry"
)

# 3. Create base map
m = folium.Map(location=[25.77, -80.25], zoom_start=13)

# 4. Add heatmap
HeatMap(data=df[['lat', 'lon']], radius=8).add_to(m)

# 5. Add shapefile overlay with precinct number tooltip
folium.GeoJson(
    gdf,
    name="Precincts",
    style_function=lambda feature: {
        'fillColor': 'red',
        'color': 'red',
        'weight': 2,
        'fillOpacity': 0.1
    },
    tooltip=folium.GeoJsonTooltip(fields=["ID"], aliases=["Precinct:"])
).add_to(m)

# 6. (Optional) Add custom legend
m.get_root().html.add_child(folium.Element(legend_html))

# 7. Save map
m.save("miami_heatmap_with_districts.html")